In [ ]:
#!/usr/bin/env python
"""
Notebook: 04_model_training.ipynb
Model Training for ECG Classification
"""

 # Model Training for ECG Classification
 
# This notebook demonstrates:
 - CNN model training
 - Autoencoder for anomaly detection
 - Hybrid model combining both approaches

# 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import sys
from pathlib import Path

sys.path.insert(0, '..')

from src.models.cnn_classifier import ECG1DCNN
from src.models.autoencoder import ECGDenoisingAutoencoder
from src.models.hybrid_model import HybridECGModel
from src.training.trainer import Trainer
from src.training.metrics import MetricsCalculator
from src.utils.reproducibility import set_seed

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(42)

print(f"Using device: {device}")

# 2. Load and Prepare Data

In [ ]:
# Generate synthetic training data for demonstration
# In production, this would load actual preprocessed beats
np.random.seed(42)
num_samples = 1000
num_classes = 5
input_dim = 187

# Generate synthetic ECG beats with class-specific patterns
X_train = []
y_train = []

for i in range(num_samples):
    class_id = np.random.randint(0, num_classes)
    
    # Base signal
    beat = np.random.randn(input_dim) * 0.1
    
    # Class-specific patterns
    if class_id == 0:  # Normal - clean with clear peak
        beat[80:110] += 1.0 * np.hanning(30)
    elif class_id == 1:  # AFib - irregular
        beat[80:110] += 0.8 * np.hanning(30)
        beat += 0.05 * np.random.randn(input_dim)
    elif class_id == 2:  # PVC - wide QRS
        beat[70:120] += 1.2 * np.hanning(50)
    elif class_id == 3:  # Bradycardia - slow
        beat[80:110] += 0.9 * np.hanning(30)
    else:  # Other - noisy
        beat += 0.2 * np.random.randn(input_dim)
        
    X_train.append(beat)
    y_train.append(class_id)

X_train = np.array(X_train).reshape(-1, 1, input_dim)
y_train = np.array(y_train)

# Split into train/val
split_idx = int(0.8 * len(X_train))
X_train, X_val = X_train[:split_idx], X_train[split_idx:]
y_train, y_val = y_train[:split_idx], y_train[split_idx:]

# Create dataloaders
train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# 3. CNN Classifier Training

In [ ]:
# Initialize model
cnn_model = ECG1DCNN(input_dim=input_dim, num_classes=num_classes).to(device)

# Training configuration
config = {
    'epochs': 50,
    'learning_rate': 0.001,
    'optimizer': 'adam',
    'loss': 'cross_entropy',
    'num_classes': num_classes,
    'log_dir': './logs/cnn_demo',
    'checkpoint_dir': './checkpoints/cnn_demo'
}

# Create trainer
trainer = Trainer(cnn_model, train_loader, val_loader, config, device)

# Train
history = trainer.train()

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], 'b-', label='Train Loss', linewidth=1.5)
axes[0].plot(history['val_loss'], 'r-', label='Val Loss', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot([m.get('accuracy', 0) for m in history['train_metrics']], 
            'b-', label='Train Acc', linewidth=1.5)
axes[1].plot([m.get('accuracy', 0) for m in history['val_metrics']], 
            'r-', label='Val Acc', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Autoencoder for Anomaly Detection

In [ ]:
# Create autoencoder (trained on normal beats only)
autoencoder = ECGDenoisingAutoencoder(input_dim=input_dim, latent_dim=32).to(device)

# Filter normal beats for autoencoder training
normal_indices = np.where(y_train == 0)[0]
X_normal = X_train[normal_indices]

normal_dataset = TensorDataset(torch.FloatTensor(X_normal), torch.FloatTensor(X_normal))
normal_loader = DataLoader(normal_dataset, batch_size=64, shuffle=True)

print(f"Training autoencoder on {len(normal_dataset)} normal beats")

# Train autoencoder
autoencoder_config = {
    'epochs': 30,
    'learning_rate': 0.001,
    'optimizer': 'adam',
    'loss': 'mse',
    'log_dir': './logs/autoencoder_demo'
}

autoencoder_trainer = Trainer(autoencoder, normal_loader, normal_loader, 
                              autoencoder_config, device)

# Custom training loop for autoencoder
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)
criterion = nn.MSELoss()

autoencoder_losses = []

for epoch in range(30):
    epoch_loss = 0
    for batch in normal_loader:
        x = batch[0].to(device)
        
        optimizer.zero_grad()
        reconstructed = autoencoder(x)
        loss = criterion(reconstructed, x)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    autoencoder_losses.append(epoch_loss / len(normal_loader))
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/30, Loss: {autoencoder_losses[-1]:.4f}")

# Plot autoencoder loss
plt.figure(figsize=(10, 5))
plt.plot(autoencoder_losses, 'b-', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss')
plt.title('Autoencoder Training')
plt.grid(True, alpha=0.3)
plt.show()

# 5. Anomaly Detection with Autoencoder

In [ ]:
# Compute reconstruction errors for validation set
autoencoder.eval()
reconstruction_errors = []

with torch.no_grad():
    for batch in val_loader:
        x = batch[0].to(device)
        reconstructed = autoencoder(x)
        error = torch.mean((reconstructed - x) ** 2, dim=(1, 2))
        reconstruction_errors.extend(error.cpu().numpy())

# Set threshold at 95th percentile of normal errors
threshold = np.percentile(reconstruction_errors, 95)
print(f"Anomaly threshold: {threshold:.4f}")

# Visualize reconstruction errors
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(reconstruction_errors, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.3f}')
axes[0].set_xlabel('Reconstruction Error')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Reconstruction Error Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(reconstruction_errors, vert=True)
axes[1].axhline(threshold, color='red', linestyle='--', linewidth=2)
axes[1].set_ylabel('Reconstruction Error')
axes[1].set_title('Reconstruction Error Box Plot')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Show example reconstructions
sample_idx = 0
sample_beat = X_val[sample_idx]

with torch.no_grad():
    reconstructed = autoencoder(torch.FloatTensor(sample_beat).unsqueeze(0).to(device))

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
time = np.arange(input_dim) / 360 * 1000

axes[0].plot(time, sample_beat[0], 'b-', linewidth=1.5)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Original Beat')
axes[0].grid(True, alpha=0.3)

axes[1].plot(time, reconstructed.cpu().numpy()[0, 0], 'r-', linewidth=1.5)
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Reconstructed Beat')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. Hybrid Model (Classifier + Autoencoder)

In [ ]:
# Initialize hybrid model
hybrid_model = HybridECGModel(input_dim=input_dim, num_classes=num_classes, latent_dim=32).to(device)

print(f"Hybrid model parameters: {sum(p.numel() for p in hybrid_model.parameters()):,}")

# Training configuration for hybrid model
hybrid_config = {
    'epochs': 40,
    'learning_rate': 0.001,
    'optimizer': 'adam',
    'loss': 'hybrid',
    'loss_kwargs': {
        'classification_weight': 0.7,
        'reconstruction_weight': 0.3
    },
    'num_classes': num_classes,
    'log_dir': './logs/hybrid_demo'
}

# Create trainer
hybrid_trainer = Trainer(hybrid_model, train_loader, val_loader, hybrid_config, device)

# Train
hybrid_history = hybrid_trainer.train()

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss comparison
axes[0].plot(history['train_loss'][:40], 'b-', label='CNN Train', linewidth=1)
axes[0].plot(history['val_loss'][:40], 'b--', label='CNN Val', linewidth=1)
axes[0].plot(hybrid_history['train_loss'], 'r-', label='Hybrid Train', linewidth=1)
axes[0].plot(hybrid_history['val_loss'], 'r--', label='Hybrid Val', linewidth=1)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
train_acc_cnn = [m.get('accuracy', 0) for m in history['train_metrics']]
val_acc_cnn = [m.get('accuracy', 0) for m in history['val_metrics']]
train_acc_hybrid = [m.get('accuracy', 0) for m in hybrid_history['train_metrics']]
val_acc_hybrid = [m.get('accuracy', 0) for m in hybrid_history['val_metrics']]

axes[1].plot(train_acc_cnn[:40], 'b-', label='CNN Train', linewidth=1)
axes[1].plot(val_acc_cnn[:40], 'b--', label='CNN Val', linewidth=1)
axes[1].plot(train_acc_hybrid, 'r-', label='Hybrid Train', linewidth=1)
axes[1].plot(val_acc_hybrid, 'r--', label='Hybrid Val', linewidth=1)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()